In [1]:
import logging

import numpy as np
from numpy.linalg import norm
from math import isfinite
from scipy.io import savemat
from copy import deepcopy
import regpy.stoprules as rules
from regpy.vecsps import DirectSum
from regpy.hilbert import L2, HmDomain
from regpy.operators import CoordinateProjection, Zero, InnerShift, OuterShift
from regpy.operators import DirectSum as opDirectSum
from operators import get_op_g_to_data
from regpy.solvers import RegularizationSetting
from regpy.solvers.nonlinear.irgnm import IrgnmCG
from regpy.solvers.nonlinear.newton import NewtonCG
from plotting import plot_exact_solution_data,plot_reco, plot_stats, init_plot_stats
from setup import setup_simulated_g
from extensions import harmonic_extension
import matplotlib.pyplot as plt
import os

# Set parameters

In [2]:
# intermediate results will be written to file names starting with output_prefix
output_prefix='example'

# abs(g) is assumed to be known outside of the nanotip.
# this is imposed as contraint. 

# total number of counts for gain and loss data
total_nr_counts = 2e9
# turn off/on all plots
do_plottings = True
# turn off/on saving of results
save_results=False
# use log(|g|) instead of |g| for darkness in phase plots of g and g_rec. Makes phase visible everywhere
plot_log_g = True
# number of modes used for evaluations of forward operator and generation of simulated data
N_data = 30
# values of N used for evaluation of the derivative of the forward operator
# This value should gradually be increased to save computation time. 
N_deriv = [4,8,16,30]
# solver type: If True, NewtonCG is used, otherwise IrgnmCG
use_NewtonCG = True

sobolev_index_phase = 2; sobolev_index_ampl = 2
IRGNM_regpar = 1e-15
IRGNM_regpar_step = 2/3
IRGNM_cgstop = 1000
NewtonCG_rho = 0.95
NewtonCG_cgmaxit = 50
# If the norm of the residual (=data-predicted data) decreases by less than minimal_residual_reduction, 
# then N is increased to the next value in N_deriv
minimal_residual_reduction = 0.98 # NewtonCG_rho**0.5
# Maximum number of Newton iterations for each value of N
max_Newton_its = 20

Setup logging and saving of results

In [4]:
current_directory=os.path.dirname(os.path.realpath(__file__))
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)-20s :: %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(current_directory,f"{output_prefix}.log"),mode='w'),
        logging.StreamHandler()
    ]
)
output_path=None
if save_results:
    output_path = os.path.join(current_directory,'data','results',output_prefix)

NameError: name '__file__' is not defined